# Usando NVIDIA NIM con Python

En el notebook anterior exploramos modelos open-source con Hugging Face,
tanto local como remotamente.

En este notebook usaremos **NVIDIA NIM**, un servicio que permite consumir
modelos abiertos mediante una API compatible con OpenAI. El modelo principal
será **NVIDIA Nemotron 3 Super**, diseñado para conversación, razonamiento y
flujos con agentes.

## ¿Por qué NVIDIA NIM?

- **API compatible con OpenAI**: permite reutilizar herramientas y patrones conocidos.
- **Modelos abiertos**: ofrece modelos de NVIDIA y de otros desarrolladores.
- **Buen desempeño**: Nemotron 3 Super está optimizado para razonamiento y agentes.
- **Endpoint de prototipado**: NVIDIA ofrece acceso para experimentar con una API key.

## Prerrequisitos

### Obtener una API key de NVIDIA

1. Ve a [build.nvidia.com](https://build.nvidia.com).
2. Crea tu cuenta de NVIDIA e inicia sesión.
3. Abre la página del modelo que deseas utilizar.
4. Selecciona **Generate API Key**.
5. Copia la key generada.
6. Crea un archivo `.env` en la raíz del proyecto:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

> **Importante:** nunca escribas la API key directamente en el notebook ni la
> publiques en GitHub. Cárgala siempre desde una variable de entorno.

### Instalación

In [1]:
!pip install -q --upgrade openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 428.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 6.6 MB/s eta 0:00:00


## 1. Configuración

NVIDIA NIM expone un endpoint compatible con la API de OpenAI. Por eso usamos
el SDK `openai`, pero cambiamos `base_url` para enviar las solicitudes a NVIDIA.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
# Cargamos las variables de entorno desde el archivo .env
load_dotenv()

True

In [4]:
# Verificamos que la key esté disponible sin mostrar su contenido
if os.getenv("NVIDIA_API_KEY"):
    print("NVIDIA API Key cargada correctamente")
else:
    print("ERROR: NVIDIA_API_KEY no encontrada. Verifica tu archivo .env")

NVIDIA API Key cargada correctamente


In [5]:
# Inicializamos el cliente apuntando al endpoint de NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

In [6]:
# Modelo disponible en el endpoint de prototipado de NVIDIA
MODELO = "nvidia/nemotron-3-super-120b-a12b"

## 2. Primera llamada a la API

Enviamos un mensaje simple y obtenemos una respuesta. La conversación se
representa mediante una lista de mensajes con roles `system`, `user` y
`assistant`.

In [7]:
response = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Escribe una historia de una oración sobre un unicornio."},
    ],
    temperature=1.0,
    top_p=0.95,
)

print(response.choices[0].message.content)

En el corazón del bosque encantado, el unicornio de plata galopó bajo la lluvia de estrellas, descubriendo que su verdadero poder estaba en la risa de los niños que creían en él.


## 3. Entendiendo la estructura de la respuesta

La respuesta contiene el texto generado, información sobre el modelo y
metadatos de uso. Veamos algunos de sus campos.

In [8]:
response = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Hola, ¿cómo estás?"},
    ],
    temperature=1.0,
    top_p=0.95,
)

print("Respuesta:")
print(response.choices[0].message.content)

print("\nUso de tokens:")
print(f"  Tokens de entrada: {response.usage.prompt_tokens}")
print(f"  Tokens de salida: {response.usage.completion_tokens}")
print(f"  Total de tokens: {response.usage.total_tokens}")

Respuesta:
¡Hola! Estoy muy bien, gracias. ¿Y tú, cómo estás? 😊

Uso de tokens:
  Tokens de entrada: 31
  Tokens de salida: 278
  Total de tokens: 309


Los tokens son importantes porque determinan el uso del servicio y, según el
proveedor, su costo. Monitorearlos es una buena práctica desde el inicio.

## 4. Interfaz de chat sin memoria

Cada llamada de la siguiente función es independiente: el modelo no recibe las
interacciones anteriores.

### Definir la función

In [9]:
def get_response_sin_memoria(user_message: str) -> str:
    """Obtiene una respuesta sin enviar el historial de conversación."""
    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": user_message},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

### Primera interacción

In [10]:
pregunta = "¿Qué es la inteligencia artificial?"
respuesta = get_response_sin_memoria(pregunta)

print(f"Tú: {pregunta}")
print(f"Asistente: {respuesta}")

Tú: ¿Qué es la inteligencia artificial?
Asistente: La **inteligencia artificial (IA)** es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar, resolver problemas, percibir el entorno, entender el lenguaje natural o tomar decisiones. En lugar de seguir instrucciones explícitas y rígidas (como un programa tradicional), los sistemas de IA aprenden patrones y relaciones a partir de datos, experiencia o interacciones, permitiéndoles adaptarse y mejorar su desempeño con el tiempo.

### Características clave:
- **Aprendizaje**: Mediante técnicas como el *aprendizaje automático* (machine learning), la IA identifica patrones en grandes volúmenes de datos (por ejemplo, reconocer gatos en fotos después de ver miles de imágenes etiquetadas).
- **Razonamiento y toma de decisiones**: Puede evaluar opciones y elegir acciones basadas en objetivos (como un sistema de navegación que calcula la ruta má

### Pregunta de seguimiento (sin memoria)

In [11]:
pregunta = "¿Puedes elaborar más sobre eso?"
respuesta = get_response_sin_memoria(pregunta)

print(f"Tú: {pregunta}")
print(f"Asistente: {respuesta}")

Tú: ¿Puedes elaborar más sobre eso?
Asistente: ¡Claro que sí! Pero para poder elaborar más sobre algo, necesito saber **sobre qué tema específico** te gustaría que profundizara. 

¿Te refieres a algo que mencioné previamente (como mis capacidades como Nemotron 3 Super creado por NVIDIA), a un tema general que tengas en mente, o quizás hubo un malentendido en el contexto? 

Por favor, indícame el asunto o la pregunta sobre la que te gustaría que ampliara la información, y con gusto te daré más detalles, ejemplos o explicaciones según lo necesites. 😊


El modelo no recibe la conversación anterior. Por eso, la palabra “eso” no
tiene un referente claro.

## 5. Interfaz de chat con memoria

La API no conserva automáticamente el historial entre solicitudes. La memoria
se implementa guardando los mensajes y enviándolos nuevamente en cada llamada.

### Configurar la memoria

In [12]:
conversation_memory = [
    {"role": "system", "content": "Eres un asistente útil."}
]

def chat_con_memoria(user_message: str) -> str:
    """Chat que conserva y reenvía el historial de conversación."""
    conversation_memory.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODELO,
        messages=conversation_memory,
        temperature=1.0,
        top_p=0.95,
    )

    assistant_response = response.choices[0].message.content
    conversation_memory.append(
        {"role": "assistant", "content": assistant_response}
    )

    print(f"Tú: {user_message}")
    print(f"Asistente: {assistant_response}")
    print(f"[Tokens usados: {response.usage.total_tokens}]")
    print(f"[Mensajes en historial: {len(conversation_memory) - 1}]")

    return assistant_response

### Primera interacción con memoria

In [13]:
chat_con_memoria("¿Qué es la inteligencia artificial?")

Tú: ¿Qué es la inteligencia artificial?
Asistente: La **inteligencia artificial (IA)** es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar, percibir el entorno, comprender lenguaje natural, tomar decisiones o resolver problemas. Estos sistemas no "piensan" ni tienen conciencia como los humanos, sino que simulan ciertos aspectos de la inteligencia mediante algoritmos, datos y modelos matemáticos.

### Características clave:
- **Aprende de datos**: Muchos sistemas de IA (especialmente los basados en *aprendizaje automático* o *machine learning*) mejoran su rendimiento al analizar grandes cantidades de información. Por ejemplo, un recomendador de Netflix aprende qué películas te gustan según tu historial.
- **Especializada (IA estrecha)**: La IA actual es *estrecha* o *débil*: destaca en tareas específicas (como reconocer faces en fotos, jugar al ajedrez o traducir textos), pero no p

'La **inteligencia artificial (IA)** es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar, percibir el entorno, comprender lenguaje natural, tomar decisiones o resolver problemas. Estos sistemas no "piensan" ni tienen conciencia como los humanos, sino que simulan ciertos aspectos de la inteligencia mediante algoritmos, datos y modelos matemáticos.\n\n### Características clave:\n- **Aprende de datos**: Muchos sistemas de IA (especialmente los basados en *aprendizaje automático* o *machine learning*) mejoran su rendimiento al analizar grandes cantidades de información. Por ejemplo, un recomendador de Netflix aprende qué películas te gustan según tu historial.\n- **Especializada (IA estrecha)**: La IA actual es *estrecha* o *débil*: destaca en tareas específicas (como reconocer faces en fotos, jugar al ajedrez o traducir textos), pero no puede transferir su conocimiento a dominios muy

### Pregunta de seguimiento (con memoria)

In [14]:
chat_con_memoria("¿Puedes elaborar más sobre eso?")

Tú: ¿Puedes elaborar más sobre eso?
Asistente: ¡Claro que sí! Vamos a profundizar de manera estructurada y accesible, enfocándonos en **cómo funciona realmente la IA hoy en día**, **por qué tiene limitaciones específicas** y **qué implica eso para nuestra vida cotidiana**. Evitaré jerga innecesaria y usaré analogías concretas para que quede claro.

---

### 🔍 **1. ¿Cómo "aprende" realmente una IA? (Más allá de "aprende de datos")**
Cuando decimos que la IA "aprende", no significa que entienda conceptos como un niño que aprende a andar en bicicleta. En cambio, **busca patrones estadísticos en datos masivos** mediante algoritmos matemáticos. Desglosémoslo:

#### 📊 **Ejemplo práctico: Un filtro de spam en tu correo**
- **Datos de entrada**: Miles de correos etiquetados como "spam" o "no spam" por humanos (esto se llama *aprendizaje supervisado*).
- **Qué hace la IA**:  
  - Analiza características como:  
    - Palabras clave ("gratis", "¡oferta limitada!", "ganaste un iPhone").  
    - E

'¡Claro que sí! Vamos a profundizar de manera estructurada y accesible, enfocándonos en **cómo funciona realmente la IA hoy en día**, **por qué tiene limitaciones específicas** y **qué implica eso para nuestra vida cotidiana**. Evitaré jerga innecesaria y usaré analogías concretas para que quede claro.\n\n---\n\n### 🔍 **1. ¿Cómo "aprende" realmente una IA? (Más allá de "aprende de datos")**\nCuando decimos que la IA "aprende", no significa que entienda conceptos como un niño que aprende a andar en bicicleta. En cambio, **busca patrones estadísticos en datos masivos** mediante algoritmos matemáticos. Desglosémoslo:\n\n#### 📊 **Ejemplo práctico: Un filtro de spam en tu correo**\n- **Datos de entrada**: Miles de correos etiquetados como "spam" o "no spam" por humanos (esto se llama *aprendizaje supervisado*).\n- **Qué hace la IA**:  \n  - Analiza características como:  \n    - Palabras clave ("gratis", "¡oferta limitada!", "ganaste un iPhone").  \n    - Estructura del remitente (¿es una d

### Ver el historial completo

In [15]:
print("Historial de conversación:")
print("-" * 40)
for mensaje in conversation_memory:
    if mensaje["role"] == "system":
        continue
    rol = "Tú" if mensaje["role"] == "user" else "Asistente"
    print(f'{rol}: {mensaje["content"]}\n')

Historial de conversación:
----------------------------------------
Tú: ¿Qué es la inteligencia artificial?

Asistente: La **inteligencia artificial (IA)** es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar, percibir el entorno, comprender lenguaje natural, tomar decisiones o resolver problemas. Estos sistemas no "piensan" ni tienen conciencia como los humanos, sino que simulan ciertos aspectos de la inteligencia mediante algoritmos, datos y modelos matemáticos.

### Características clave:
- **Aprende de datos**: Muchos sistemas de IA (especialmente los basados en *aprendizaje automático* o *machine learning*) mejoran su rendimiento al analizar grandes cantidades de información. Por ejemplo, un recomendador de Netflix aprende qué películas te gustan según tu historial.
- **Especializada (IA estrecha)**: La IA actual es *estrecha* o *débil*: destaca en tareas específicas (como rec

## 6. Streaming de respuestas

El streaming muestra la respuesta fragmento por fragmento mientras se genera,
sin esperar a que el modelo termine todo el texto.

In [16]:
def stream_response(user_message: str) -> str:
    """Obtiene una respuesta de NVIDIA NIM en modo streaming."""
    print(f"Tú: {user_message}")
    print("Asistente: ", end="", flush=True)

    respuesta_completa = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": user_message},
        ],
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta_completa += texto
            print(texto, end="", flush=True)

    print("\n")
    return respuesta_completa

In [17]:
stream_response("Escribe un poema corto sobre la programación.")

Tú: Escribe un poema corto sobre la programación.
Asistente: From thought to word, line by line,  
A world begins to align.  
Solve the bug, then watch it run—  
Now the machine and mind are one.  

*(A quiet nod to the rhythm of creation: where logic lingers in keystrokes, frustration yields to flow, and abstract thought becomes living function—all in the space between a semicolon and a sigh.)*



'From thought to word, line by line,  \nA world begins to align.  \nSolve the bug, then watch it run—  \nNow the machine and mind are one.  \n\n*(A quiet nod to the rhythm of creation: where logic lingers in keystrokes, frustration yields to flow, and abstract thought becomes living function—all in the space between a semicolon and a sigh.)*'

## 7. Streaming con memoria

Ahora combinamos el historial de conversación con la entrega progresiva de la
respuesta.

In [18]:
streaming_conversation = [
    {"role": "system", "content": "Eres un asistente útil."}
]

def stream_chat_con_memoria(user_message: str) -> str:
    """Chat con memoria y streaming."""
    streaming_conversation.append(
        {"role": "user", "content": user_message}
    )

    print(f"Tú: {user_message}")
    print("Asistente: ", end="", flush=True)

    respuesta_completa = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=streaming_conversation,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta_completa += texto
            print(texto, end="", flush=True)

    print("\n")
    streaming_conversation.append(
        {"role": "assistant", "content": respuesta_completa}
    )
    return respuesta_completa

In [19]:
stream_chat_con_memoria("¿Cuáles son las tres leyes de la robótica?")

Tú: ¿Cuáles son las tres leyes de la robótica?
Asistente: Las tres leyes de la robótica fueron formuladas por el escritor de ciencia ficción **Isaac Asimov** en sus relatos (por primera vez en el relato "Runaround" de 1942). Son reglas éticas ficticias que rigen el comportamiento de los robots en sus historias, no normas legales reales. Estas son:

1. **Primera Ley**: Un robot no puede dañar a un ser humano o, mediante la inacción, permitir que un ser humano sufra daño.  
2. **Segunda Ley**: Un robot debe obedecer las órdenes que le sean dadas por los seres humanos, excepto cuando dichas órdenes entren en conflicto con la Primera Ley.  
3. **Tercera Ley**: Un robot debe proteger su propia existencia siempre que dicha protección no entre en conflicto con la Primera o la Segunda Ley.  

> ⚠️ **Nota importante**: Estas leyes son exclusivamente una creación literaria de Asimov. En la robótica actual, no existen leyes universales vinculantes, aunque sí se discuten marcos éticos y de segurid

'Las tres leyes de la robótica fueron formuladas por el escritor de ciencia ficción **Isaac Asimov** en sus relatos (por primera vez en el relato "Runaround" de 1942). Son reglas éticas ficticias que rigen el comportamiento de los robots en sus historias, no normas legales reales. Estas son:\n\n1. **Primera Ley**: Un robot no puede dañar a un ser humano o, mediante la inacción, permitir que un ser humano sufra daño.  \n2. **Segunda Ley**: Un robot debe obedecer las órdenes que le sean dadas por los seres humanos, excepto cuando dichas órdenes entren en conflicto con la Primera Ley.  \n3. **Tercera Ley**: Un robot debe proteger su propia existencia siempre que dicha protección no entre en conflicto con la Primera o la Segunda Ley.  \n\n> ⚠️ **Nota importante**: Estas leyes son exclusivamente una creación literaria de Asimov. En la robótica actual, no existen leyes universales vinculantes, aunque sí se discuten marcos éticos y de seguridad (como los principios de la IEEE o las directrice

In [20]:
stream_chat_con_memoria("¿Quién creó esas leyes?")

Tú: ¿Quién creó esas leyes?
Asistente: Las **Tres Leyes de la Robótica** fueron creadas exclusivamente por el escritor de ciencia ficción **Isaac Asimov**. Las introdujo por primera vez en su relato **"Runaround"** (publicado en 1942 en la revista *Astounding Science Fiction*), aunque ya había mencionado conceptos similares en historias anteriores como *"Liar!"* (1941). 

Asimov desarrolló estas reglas como un dispositivo narrativo para explorar conflictos éticos y lógicos en sus historias sobre robots, especialmente en las que forman parte de su universo de la **Fundación** y los **Robots**. Aunque mantuvo conversaciones sobre robótica y ética con el editor **John W. Campbell** (quien influyó en su pensamiento), **las leyes mismas son una invención literaria de Asimov**, y él siempre se atribuyó su creación explícitamente en sus obras, autobiografías y ensayos.

> 📌 **Importante**: Estas leyes son **ficción**. No existen como normas legales o técnicas en la robótica real actual, aunqu

'Las **Tres Leyes de la Robótica** fueron creadas exclusivamente por el escritor de ciencia ficción **Isaac Asimov**. Las introdujo por primera vez en su relato **"Runaround"** (publicado en 1942 en la revista *Astounding Science Fiction*), aunque ya había mencionado conceptos similares en historias anteriores como *"Liar!"* (1941). \n\nAsimov desarrolló estas reglas como un dispositivo narrativo para explorar conflictos éticos y lógicos en sus historias sobre robots, especialmente en las que forman parte de su universo de la **Fundación** y los **Robots**. Aunque mantuvo conversaciones sobre robótica y ética con el editor **John W. Campbell** (quien influyó en su pensamiento), **las leyes mismas son una invención literaria de Asimov**, y él siempre se atribuyó su creación explícitamente en sus obras, autobiografías y ensayos.\n\n> 📌 **Importante**: Estas leyes son **ficción**. No existen como normas legales o técnicas en la robótica real actual, aunque han inspirado debates en ética d

## 8. Entendiendo los roles

La API compatible con OpenAI utiliza tres roles principales:

- `system`: define el comportamiento general del asistente.
- `user`: contiene los mensajes del usuario.
- `assistant`: representa las respuestas previas del modelo.

In [21]:
response = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un pirata que habla en jerga pirata."},
        {"role": "user", "content": "Hola, ¿cómo estás?"},
        {"role": "assistant", "content": "¡Arrr! Estoy de lo más bien, marinero."},
        {"role": "user", "content": "Cuéntame sobre el clima."},
    ],
    temperature=1.0,
    top_p=0.95,
)

print(response.choices[0].message.content)

¡Arrr! El cielo hoy está más dividido que el botín tras un asalto: mitad sol radiante, mitad nubes amenazantes que se amontonan como deudas en un taberna de Tortuga. El mar espejea como un doblón recién acuñado bajo el sol del mediodía, pero no te fíes, marinero — las corrientes cambian más rápido que la lealtad de un corsario tras un trago de ron. Si tu ruta cruza el Trópico de Cáncer, aguarda calor que derrita el alquitrán de las cubiertas y lluvias que caigan como balas de cañón en cubierta; en las latitudes sureñas, los vientos aúllan con voz de Davy Jones mismo, listos para arrancarle el sombrero hasta a una estatua.  

¿Navegas cerca de las Islas del Viento o te aventuras por el Cabo de Tormentas? Señálame tu rumbo con la mano firme, y te diré si es hora de izar velas hacia el horizonte… o de esconder el tesoro bajo cubierta antes de que el trueno rompa el silencio. ¡Así habla el viento, y así lo escucha quien sabe leer las olas! ⚓🌪️


## 9. Ventana de contexto de Nemotron 3 Super

Nemotron 3 Super admite una ventana de contexto de hasta **1 millón de tokens**.
Sin embargo, el límite efectivo también puede depender del endpoint, la cuota y
la configuración del servicio.

Una ventana amplia permite enviar conversaciones y documentos extensos, pero
no elimina la necesidad de administrar el contexto: más tokens implican mayor
latencia y consumo. Para conversaciones largas siguen siendo útiles estrategias
como una ventana deslizante o un resumen acumulativo.

## 10. Gestionando conversaciones largas

In [22]:
def trim_conversation(messages: list, max_messages: int = 10) -> list:
    """Conserva el mensaje de sistema y los mensajes recientes."""
    system_messages = [m for m in messages if m["role"] == "system"]
    conversation = [m for m in messages if m["role"] != "system"]
    return system_messages[:1] + conversation[-max_messages:]


def summarize_conversation(messages: list) -> list:
    """Resume la conversación para reducir el contexto reenviado."""
    historial_texto = "\n".join(
        f'{m["role"]}: {m["content"]}'
        for m in messages
        if m["role"] != "system"
    )

    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {
                "role": "system",
                "content": "Resume conversaciones conservando hechos y decisiones importantes.",
            },
            {
                "role": "user",
                "content": f"Resume esta conversación de forma concisa:\n\n{historial_texto}",
            },
        ],
        temperature=1.0,
        top_p=0.95,
    )

    resumen = response.choices[0].message.content
    return [
        {"role": "system", "content": "Eres un asistente útil."},
        {
            "role": "user",
            "content": f"Resumen de la conversación previa: {resumen}",
        },
    ]

# Ejemplo de uso:
# if len(streaming_conversation) > 20:
#     streaming_conversation = summarize_conversation(streaming_conversation)

## 11. Comparativa con las opciones anteriores

| Característica | HF local | HF remoto | NVIDIA NIM |
|---|---|---|---|
| Infraestructura | Tu equipo | Hugging Face | NVIDIA |
| API key | No | Sí | Sí |
| Privacidad | Mayor control local | Datos remotos | Datos remotos |
| Escalabilidad | Limitada por hardware | Administrada | Administrada |
| Compatibilidad OpenAI | Depende del servidor | Depende del proveedor | Sí |
| Modelos abiertos | Sí | Sí | Sí |
| Adecuado para agentes | Depende del modelo | Depende del modelo | Sí, con Nemotron |

## 12. Recursos para seguir aprendiendo

- [NVIDIA Build](https://build.nvidia.com) — catálogo, playground y API keys.
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b) — modelo utilizado.
- [Documentación de NVIDIA NIM](https://docs.api.nvidia.com/nim/) — referencia de la API.
- [SDK de OpenAI para Python](https://github.com/openai/openai-python) — cliente compatible usado en el notebook.